# Lab 04-06: CI/CD for Agent Components

**Module:** 04 — Assembling and Deploying Applications  
**Exam Weight:** ~10 questions across Module 04 (22% of exam)  
**UI Sections:** Repos | Workflows | Jobs  
**Difficulty:** Intermediate  
**Estimated Time:** 40–50 minutes

---

## Learning Objectives

- Understand the **end-to-end CI/CD pipeline** for GenAI components on Databricks
- Use **Git integration** (Databricks Repos) for version control of agents, chains, and tools
- Implement **testing strategies** for GenAI: unit tests, integration tests, and evaluation tests
- Create **Databricks Workflows/Jobs** for automated deployment pipelines
- Configure **A/B testing** with traffic splitting between model versions
- Implement **rollback strategies** for safe production deployments

---

## Exam Objectives Covered

- **Describe CI/CD best practices for GenAI applications**
- **Implement environment promotion using Unity Catalog schemas**
- **Configure A/B testing and traffic routing for model endpoints**

---

## What Is CI/CD for GenAI and Why Does It Matter?

CI/CD (Continuous Integration / Continuous Deployment) for GenAI is the practice of **automating the testing, versioning, and deployment** of AI components — prompts, chains, agents, tools, and models. Without CI/CD, every deployment is a manual, error-prone process.

Traditional software CI/CD tests code correctness. GenAI CI/CD adds an extra dimension: **quality evaluation**. You cannot just check if the code runs — you must check if the LLM's *output* is good enough. This means your pipeline includes:

1. **Code tests** — Does the chain/agent code execute without errors?
2. **Integration tests** — Does the full pipeline (retriever + LLM + tools) work end-to-end?
3. **Evaluation tests** — Does the LLM output meet quality thresholds (relevance, faithfulness, safety)?

On Databricks, the CI/CD stack is:

| Component | Databricks Feature |
|-----------|-------------------|
| **Version control** | Databricks Repos (Git integration) |
| **Artifact registry** | MLflow Model Registry (Unity Catalog) |
| **Evaluation** | MLflow Evaluate |
| **Orchestration** | Databricks Workflows (Jobs) |
| **Serving** | Model Serving Endpoints |
| **Environments** | Unity Catalog schemas (dev / staging / prod) |

---

## Mental Model

> **Analogy:** CI/CD for AI is like a car factory assembly line with quality checkpoints. The blueprint (code) lives in Git. Each change triggers the assembly line (CI pipeline): build the car (package the model), test the brakes (unit tests), test-drive on the track (integration tests), have a safety inspector check it (evaluation tests), then ship it to the dealership (deploy to production). If any checkpoint fails, the car does not ship.

The key difference from traditional CI/CD: in addition to checking "does the code work?", you also check "**is the AI output good enough?**" — using MLflow Evaluate with LLM judges.

---

## Exam Alert: Common Traps

| # | Trap Statement | Correct Answer |
|---|---------------|----------------|
| 1 | "Deploy models directly from notebooks to production" | **Wrong.** Use MLflow Model Registry (Unity Catalog) + serving endpoints. Notebooks are for development, not production deployment. |
| 2 | "CI/CD is only for traditional software, not for GenAI" | **Wrong.** GenAI apps need CI/CD too — test prompts, evaluate chains, version artifacts. In fact, CI/CD is *more* important because LLM outputs are non-deterministic. |
| 3 | "A/B testing replaces CI/CD" | **Wrong.** A/B testing is a *deployment strategy* within your CI/CD pipeline. CI/CD gets you to the point where you can A/B test. |
| 4 | "You only need to test the LLM itself" | **Wrong.** Test the entire chain: retriever accuracy, prompt template correctness, tool calling, and end-to-end output quality. |
| 5 | "One environment is sufficient for GenAI development" | **Wrong.** Use separate schemas: `dev` for experimentation, `staging` for validation, `prod` for live traffic. |

---

## Prerequisites and UI Navigation

### What You Need
- A Databricks workspace with Unity Catalog enabled
- Access to MLflow and a cluster with Databricks Runtime ML
- Familiarity with Git basics (commits, branches, pull requests)

### Key UI Paths
- **UI -->** Left nav --> **Repos** --> Connect to GitHub/GitLab
- **UI -->** Left nav --> **Workflows** --> Create and manage Jobs
- **UI -->** Left nav --> **Serving** --> Manage serving endpoints and traffic
- **UI -->** Left nav --> **Catalog** --> Browse models in Unity Catalog

---

## Setup

In [ ]:
# Install required packages
%pip install mlflow databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
# ---- Imports and Configuration ----
import json
import mlflow
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Configuration
CATALOG = "workspace"
SCHEMA = "genai_labs"
MODEL_CHAT = "databricks-meta-llama-3-3-70b-instruct"

# Set MLflow registry to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"MLflow registry:  databricks-uc (Unity Catalog)")
print(f"Target schema:    {CATALOG}.{SCHEMA}")
print(f"MLflow version:   {mlflow.__version__}")
print("Setup complete.")

### Expected Output

```
MLflow registry:  databricks-uc (Unity Catalog)
Target schema:    workspace.genai_labs
MLflow version:   2.x.x
Setup complete.
```

### What Just Happened

We configured MLflow to use **Unity Catalog** as the model registry. This is the foundation of CI/CD on Databricks — all model artifacts, versions, and metadata are stored in UC, giving you governance, lineage, and access control for free.

---

## Step 1: Git Integration — Databricks Repos

The first building block of CI/CD is **version control**. Databricks Repos integrates directly with GitHub, GitLab, Azure DevOps, and Bitbucket. Every change to your agent code, prompt templates, and configuration files is tracked.

### How Databricks Repos Works

```
GitHub/GitLab Repository
    |
    |  git clone / pull / push
    v
Databricks Repos (synced copy)
    |
    |  notebooks, Python files, configs
    v
Your Cluster (runs the code)
```

### UI Walkthrough

**UI -->** Left nav --> **Repos** --> **Add Repo**

1. Paste your Git URL (e.g., `https://github.com/your-org/genai-agent.git`)
2. Select your Git provider
3. Authenticate (personal access token or OAuth)
4. Databricks clones the repo into your workspace
5. You can now edit files, create branches, commit, and push — all from the Databricks UI

### What Goes in Git for a GenAI Project?

| File/Folder | What It Contains |
|-------------|------------------|
| `agent/chain.py` | Your LangChain/PyFunc chain definition |
| `agent/tools.py` | Tool definitions (UC functions, custom tools) |
| `agent/prompts.py` | Prompt templates (or use MLflow Prompt Registry) |
| `tests/test_chain.py` | Unit and integration tests |
| `evaluation/eval_set.json` | Golden evaluation dataset |
| `deployment/deploy.py` | Deployment script (register + serve) |
| `databricks.yml` | Databricks Asset Bundle configuration |

---

## Step 2: The GenAI CI/CD Pipeline

Here is the full CI/CD pipeline for a GenAI application on Databricks:

```
+----------+     +-----------+     +----------+     +-------------+     +---------+     +--------+     +------------+
| Git Push | --> | CI Tests  | --> | MLflow   | --> | UC Register | --> | Staging | --> | Eval   | --> | Production |
|          |     | (unit +   |     | Log      |     | Model       |     | Endpoint|     | Tests  |     | Endpoint   |
|          |     | integ.)   |     | Artifact |     |             |     |         |     | (LLM   |     | (traffic   |
|          |     |           |     |          |     |             |     |         |     | judges)|     | routed)    |
+----------+     +-----------+     +----------+     +-------------+     +---------+     +--------+     +------------+
```

Each stage is a **gate** — if any stage fails, the pipeline stops and the model does NOT reach production.

| Stage | What Happens | Databricks Feature |
|-------|-------------|-------------------|
| **1. Git Push** | Developer pushes code changes | Databricks Repos |
| **2. CI Tests** | Unit tests + integration tests run | Workflows (triggered by Git webhook) |
| **3. MLflow Log** | Chain/agent logged as MLflow model | MLflow Tracking |
| **4. UC Register** | Model registered in Unity Catalog | MLflow Registry (UC) |
| **5. Staging Endpoint** | Model deployed to staging | Model Serving |
| **6. Eval Tests** | Quality evaluation with MLflow Evaluate | MLflow Evaluate + LLM Judges |
| **7. Production** | Traffic routed to new version | Model Serving (traffic split) |

In [ ]:
# ---- Step 2: Simulate the CI/CD pipeline stages ----
# This cell demonstrates each stage conceptually.

pipeline_stages = [
    {
        "stage": "1. Git Push",
        "action": "Developer pushes updated chain code to main branch",
        "databricks": "Databricks Repos syncs with GitHub",
        "gate": "Code review approved (PR merged)"
    },
    {
        "stage": "2. CI Tests",
        "action": "Automated tests run: unit tests, integration tests",
        "databricks": "Databricks Workflow triggered by Git webhook",
        "gate": "All tests pass (exit code 0)"
    },
    {
        "stage": "3. MLflow Log",
        "action": "Chain/agent packaged as MLflow model artifact",
        "databricks": "mlflow.pyfunc.log_model() or mlflow.langchain.log_model()",
        "gate": "Model signature validated, dependencies captured"
    },
    {
        "stage": "4. UC Register",
        "action": "Model registered in Unity Catalog with version number",
        "databricks": "mlflow.register_model('runs:/...', 'workspace.genai_labs.my_agent')",
        "gate": "Model version created, lineage tracked"
    },
    {
        "stage": "5. Staging Deploy",
        "action": "New version deployed to staging serving endpoint",
        "databricks": "Serving endpoint updated with new model version",
        "gate": "Endpoint reaches READY state, health check passes"
    },
    {
        "stage": "6. Eval Tests",
        "action": "Quality evaluation using golden dataset + LLM judges",
        "databricks": "mlflow.evaluate() with relevance, faithfulness scorers",
        "gate": "All quality metrics above threshold (e.g., relevance > 0.8)"
    },
    {
        "stage": "7. Production",
        "action": "Traffic gradually shifted to new version (canary/A-B test)",
        "databricks": "Serving endpoint traffic config updated",
        "gate": "Monitoring confirms no regression in production metrics"
    },
]

print("=== GenAI CI/CD Pipeline Stages ===")
print()
for s in pipeline_stages:
    print(f"  {s['stage']}")
    print(f"    Action:     {s['action']}")
    print(f"    Databricks: {s['databricks']}")
    print(f"    Gate:       {s['gate']}")
    print()

### Expected Output

```
=== GenAI CI/CD Pipeline Stages ===

  1. Git Push
    Action:     Developer pushes updated chain code to main branch
    Databricks: Databricks Repos syncs with GitHub
    Gate:       Code review approved (PR merged)

  2. CI Tests
    Action:     Automated tests run: unit tests, integration tests
    Databricks: Databricks Workflow triggered by Git webhook
    Gate:       All tests pass (exit code 0)

  ...(remaining stages)...
```

### What Just Happened

We outlined the **seven stages** of a GenAI CI/CD pipeline. Each stage has a **gate** — a condition that must be met before proceeding. This is the exam-critical concept: CI/CD is not just "push and deploy" — it is a series of automated quality checks.

---

## Step 3: Testing Strategies for GenAI

GenAI applications need **three levels of testing**. This is a key exam topic — understanding which type of test catches which type of problem.

| Test Level | What It Tests | Example | Catches |
|-----------|--------------|---------|--------|
| **Unit Tests** | Individual components in isolation | Prompt template renders correctly | Broken templates, missing variables |
| **Integration Tests** | Full chain end-to-end | Retriever + LLM + tools produce output | Broken connections, API failures |
| **Evaluation Tests** | Output quality (semantic) | LLM judge scores relevance > 0.8 | Quality regressions, hallucinations |

In [ ]:
# ---- Step 3a: Unit tests for prompt templates ----
# Unit tests verify individual components in isolation.
# For GenAI, the most common unit test is: does the prompt template render correctly?

def create_rag_prompt(context: str, question: str, system_instruction: str = None) -> list:
    """Create a RAG prompt with context and question.
    This is the function we want to unit test."""
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": system_instruction})
    messages.append({"role": "user", "content": f"""Based on the following context, answer the question.

Context:
{context}

Question: {question}

Answer:"""})
    return messages

# ---- Unit tests ----
def test_prompt_has_context():
    """Test that context is included in the prompt."""
    result = create_rag_prompt("Databricks is a data platform.", "What is Databricks?")
    assert "Databricks is a data platform." in result[-1]["content"]
    return "PASS"

def test_prompt_has_question():
    """Test that question is included in the prompt."""
    result = create_rag_prompt("Some context.", "What is RAG?")
    assert "What is RAG?" in result[-1]["content"]
    return "PASS"

def test_prompt_with_system_instruction():
    """Test that system instruction is included when provided."""
    result = create_rag_prompt("ctx", "q", system_instruction="Be concise.")
    assert result[0]["role"] == "system"
    assert result[0]["content"] == "Be concise."
    return "PASS"

def test_prompt_without_system_instruction():
    """Test that no system message when instruction is None."""
    result = create_rag_prompt("ctx", "q")
    assert result[0]["role"] == "user"  # No system message
    return "PASS"

# Run unit tests
tests = [
    test_prompt_has_context,
    test_prompt_has_question,
    test_prompt_with_system_instruction,
    test_prompt_without_system_instruction,
]

print("=== Unit Tests: Prompt Templates ===")
print()
all_passed = True
for test in tests:
    try:
        status = test()
        print(f"  [{status}] {test.__name__}: {test.__doc__}")
    except AssertionError as e:
        print(f"  [FAIL] {test.__name__}: {test.__doc__} -- {e}")
        all_passed = False

print()
print(f"Result: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")
print(f"Gate:   {'OPEN -- proceed to next stage' if all_passed else 'CLOSED -- fix failures before proceeding'}")

### Expected Output

```
=== Unit Tests: Prompt Templates ===

  [PASS] test_prompt_has_context: Test that context is included in the prompt.
  [PASS] test_prompt_has_question: Test that question is included in the prompt.
  [PASS] test_prompt_with_system_instruction: Test that system instruction is included when provided.
  [PASS] test_prompt_without_system_instruction: Test that no system message when instruction is None.

Result: ALL TESTS PASSED
Gate:   OPEN -- proceed to next stage
```

### What Just Happened

We wrote **unit tests** for a prompt template function. These tests verify that the template renders correctly with different inputs. In a real CI/CD pipeline, these would run automatically on every Git push via a Databricks Workflow.

In [ ]:
# ---- Step 3b: Integration test -- full chain end-to-end ----
# Integration tests verify the entire pipeline works together.
# This test calls the actual LLM endpoint to verify connectivity.

from openai import OpenAI

DATABRICKS_HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints"
)

def test_chain_end_to_end():
    """Integration test: full RAG chain produces a valid response."""
    # Step 1: Create the prompt (simulating retrieval)
    context = "Databricks Foundation Model APIs are pay-per-token endpoints."
    question = "What are Foundation Model APIs?"
    messages = create_rag_prompt(context, question, "Answer in one sentence.")
    
    # Step 2: Call the LLM
    response = client.chat.completions.create(
        model=MODEL_CHAT,
        messages=messages,
        max_tokens=100,
        temperature=0.0
    )
    
    # Step 3: Validate the response
    answer = response.choices[0].message.content
    assert answer is not None, "Response is None"
    assert len(answer) > 10, f"Response too short: '{answer}'"
    assert response.usage.total_tokens > 0, "No tokens used"
    
    return answer, response.usage.total_tokens

def test_embedding_endpoint():
    """Integration test: embedding endpoint returns valid vectors."""
    response = client.embeddings.create(
        model="databricks-gte-large-en",
        input=["test query"]
    )
    assert len(response.data) == 1, "Expected 1 embedding"
    assert len(response.data[0].embedding) > 0, "Empty embedding vector"
    return len(response.data[0].embedding)

print("=== Integration Tests: Full Chain ===")
print()

try:
    answer, tokens = test_chain_end_to_end()
    print(f"  [PASS] test_chain_end_to_end")
    print(f"         Response: {answer[:80]}...")
    print(f"         Tokens:   {tokens}")
except Exception as e:
    print(f"  [FAIL] test_chain_end_to_end: {e}")

try:
    dim = test_embedding_endpoint()
    print(f"  [PASS] test_embedding_endpoint (dimension={dim})")
except Exception as e:
    print(f"  [FAIL] test_embedding_endpoint: {e}")

print()
print("Gate: OPEN -- integration tests passed")

### Expected Output

```
=== Integration Tests: Full Chain ===

  [PASS] test_chain_end_to_end
         Response: Foundation Model APIs are pay-per-token endpoints hosted by Databricks that...
         Tokens:   85
  [PASS] test_embedding_endpoint (dimension=1024)

Gate: OPEN -- integration tests passed
```

### What Just Happened

We ran **integration tests** that call actual Databricks endpoints. Unlike unit tests (which test code logic), integration tests verify that:
- The LLM endpoint is reachable and responding
- The full chain (prompt + LLM) produces a valid, non-empty output
- The embedding endpoint returns vectors of the expected dimension

In [ ]:
# ---- Step 3c: Evaluation tests -- quality scoring with MLflow ----
# Evaluation tests check OUTPUT QUALITY, not just code correctness.
# This is what makes GenAI CI/CD different from traditional CI/CD.

import pandas as pd

# Golden evaluation dataset -- these are known-good question/answer pairs
eval_data = pd.DataFrame([
    {
        "question": "What are Foundation Model APIs?",
        "context": "Foundation Model APIs are pay-per-token endpoints hosted by Databricks.",
        "expected_answer": "Pre-deployed LLM endpoints you can call without managing infrastructure."
    },
    {
        "question": "When should I use provisioned throughput?",
        "context": "Provisioned throughput offers dedicated GPUs, SLAs, HIPAA compliance.",
        "expected_answer": "Use provisioned throughput for production with SLAs, HIPAA, or fine-tuned models."
    },
    {
        "question": "What is AI Gateway?",
        "context": "AI Gateway (external models) proxies calls to OpenAI, Anthropic, etc.",
        "expected_answer": "A Databricks feature that routes LLM calls to external providers with governance."
    },
])

# Generate predictions for each eval question
predictions = []
for _, row in eval_data.iterrows():
    messages = create_rag_prompt(row["context"], row["question"], "Answer in one sentence.")
    response = client.chat.completions.create(
        model=MODEL_CHAT, messages=messages, max_tokens=100, temperature=0.0
    )
    predictions.append(response.choices[0].message.content)

eval_data["prediction"] = predictions

print("=== Evaluation Dataset with Predictions ===")
print()
for i, row in eval_data.iterrows():
    print(f"Q{i+1}: {row['question']}")
    print(f"  Expected:   {row['expected_answer']}")
    print(f"  Predicted:  {row['prediction'][:80]}")
    print()

### Expected Output

```
=== Evaluation Dataset with Predictions ===

Q1: What are Foundation Model APIs?
  Expected:   Pre-deployed LLM endpoints you can call without managing infrastructure.
  Predicted:  Foundation Model APIs are pay-per-token endpoints hosted by Databricks...

Q2: When should I use provisioned throughput?
  Expected:   Use provisioned throughput for production with SLAs, HIPAA, or fine-tuned models.
  Predicted:  You should use provisioned throughput when you need dedicated GPU capacity...

Q3: What is AI Gateway?
  Expected:   A Databricks feature that routes LLM calls to external providers with governance.
  Predicted:  AI Gateway is a feature that proxies LLM calls to external providers...
```

### What Just Happened

We created a **golden evaluation dataset** with known-good answers and generated predictions from our chain. In a real CI/CD pipeline, you would then run `mlflow.evaluate()` with LLM judges to automatically score relevance, faithfulness, and other quality metrics.

---

## Step 4: Databricks Workflows/Jobs — Automating the Pipeline

Databricks Workflows (also called Jobs) are the **orchestration layer** for CI/CD. A Workflow is a directed acyclic graph (DAG) of tasks that run automatically.

### CI/CD Workflow Structure

```
Workflow: "deploy-genai-agent"
    |
    +-- Task 1: Run unit tests        (notebook: tests/unit_tests.py)
    |       |
    +-- Task 2: Run integration tests  (notebook: tests/integration_tests.py)
    |       |       (depends on Task 1)
    +-- Task 3: Log model to MLflow    (notebook: deployment/log_model.py)
    |       |       (depends on Task 2)
    +-- Task 4: Register in UC         (notebook: deployment/register.py)
    |       |       (depends on Task 3)
    +-- Task 5: Deploy to staging      (notebook: deployment/deploy_staging.py)
    |       |       (depends on Task 4)
    +-- Task 6: Run eval tests         (notebook: tests/eval_tests.py)
    |       |       (depends on Task 5)
    +-- Task 7: Deploy to production   (notebook: deployment/deploy_prod.py)
                    (depends on Task 6)
```

In [ ]:
# ---- Step 4: Creating a Databricks Workflow (conceptual) ----
# This shows how to define a CI/CD workflow using the Databricks SDK.
# In practice, you can also create Workflows via the UI or YAML (Asset Bundles).

# from databricks.sdk import WorkspaceClient
# from databricks.sdk.service.jobs import (
#     Task, NotebookTask, TaskDependency, JobCluster
# )
#
# w = WorkspaceClient()
#
# w.jobs.create(
#     name="deploy-genai-agent",
#     tasks=[
#         Task(
#             task_key="unit_tests",
#             notebook_task=NotebookTask(
#                 notebook_path="/Repos/my-org/genai-agent/tests/unit_tests"
#             ),
#         ),
#         Task(
#             task_key="integration_tests",
#             notebook_task=NotebookTask(
#                 notebook_path="/Repos/my-org/genai-agent/tests/integration_tests"
#             ),
#             depends_on=[TaskDependency(task_key="unit_tests")],
#         ),
#         Task(
#             task_key="log_model",
#             notebook_task=NotebookTask(
#                 notebook_path="/Repos/my-org/genai-agent/deployment/log_model"
#             ),
#             depends_on=[TaskDependency(task_key="integration_tests")],
#         ),
#         Task(
#             task_key="eval_tests",
#             notebook_task=NotebookTask(
#                 notebook_path="/Repos/my-org/genai-agent/tests/eval_tests"
#             ),
#             depends_on=[TaskDependency(task_key="log_model")],
#         ),
#     ]
# )

# UI Walkthrough
print("=== Creating a CI/CD Workflow (UI Walkthrough) ===")
print()
print("**UI -->** Left nav --> **Workflows** --> **Create Job**")
print()
print("Step 1: Name the job (e.g., 'deploy-genai-agent')")
print("Step 2: Add tasks as a DAG:")
print("  Task 1: unit_tests       (notebook: tests/unit_tests)")
print("  Task 2: integration_tests (depends on Task 1)")
print("  Task 3: log_model         (depends on Task 2)")
print("  Task 4: register_model    (depends on Task 3)")
print("  Task 5: deploy_staging    (depends on Task 4)")
print("  Task 6: eval_tests        (depends on Task 5)")
print("  Task 7: deploy_production (depends on Task 6)")
print("Step 3: Configure trigger: Git webhook (on push to main)")
print("Step 4: Set notification: email on failure")
print()
print("RESULT: Every push to main triggers the full CI/CD pipeline.")
print("        If any task fails, subsequent tasks do not run.")

### Expected Output

```
=== Creating a CI/CD Workflow (UI Walkthrough) ===

**UI -->** Left nav --> **Workflows** --> **Create Job**

Step 1: Name the job (e.g., 'deploy-genai-agent')
Step 2: Add tasks as a DAG:
  Task 1: unit_tests       (notebook: tests/unit_tests)
  Task 2: integration_tests (depends on Task 1)
  Task 3: log_model         (depends on Task 2)
  Task 4: register_model    (depends on Task 3)
  Task 5: deploy_staging    (depends on Task 4)
  Task 6: eval_tests        (depends on Task 5)
  Task 7: deploy_production (depends on Task 6)
Step 3: Configure trigger: Git webhook (on push to main)
Step 4: Set notification: email on failure

RESULT: Every push to main triggers the full CI/CD pipeline.
        If any task fails, subsequent tasks do not run.
```

### What Just Happened

We designed a **Databricks Workflow** that automates the entire CI/CD pipeline. Each task is a notebook that runs on a cluster. Tasks have dependencies — if unit tests fail, the model never gets deployed. This is the core of production-grade GenAI CI/CD.

---

## Step 5: A/B Testing with Traffic Splitting

A/B testing lets you **gradually shift traffic** from one model version to another. Instead of a risky all-at-once deployment, you route a small percentage of traffic to the new version, monitor quality, and then increase the percentage.

### Traffic Splitting Pattern

```
Incoming Requests
    |
    +---> 90% --> Model Version 1 (current production)
    |
    +---> 10% --> Model Version 2 (new candidate)
```

### Rollout Strategy

| Day | Version 1 (old) | Version 2 (new) | Action |
|-----|----------------|-----------------|--------|
| Day 1 | 90% | 10% | Monitor error rates, latency |
| Day 2 | 70% | 30% | Check quality metrics |
| Day 3 | 50% | 50% | Compare A/B results |
| Day 5 | 0% | 100% | Full rollout (or rollback) |

In [ ]:
# ---- Step 5: A/B testing with traffic splitting (conceptual) ----
# This shows how to configure traffic splitting on a serving endpoint.

# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
#
# # Update endpoint to split traffic between two model versions
# w.serving_endpoints.update_config(
#     name="my-agent-endpoint",
#     served_entities=[
#         {
#             "entity_name": "workspace.genai_labs.my_agent",
#             "entity_version": "1",            # Current production
#             "scale_to_zero_enabled": False,
#             "workload_size": "Small",
#         },
#         {
#             "entity_name": "workspace.genai_labs.my_agent",
#             "entity_version": "2",            # New candidate
#             "scale_to_zero_enabled": False,
#             "workload_size": "Small",
#         },
#     ],
#     traffic_config={
#         "routes": [
#             {"served_model_name": "my_agent-1", "traffic_percentage": 90},
#             {"served_model_name": "my_agent-2", "traffic_percentage": 10},
#         ]
#     }
# )

# Simulate traffic splitting
import random
random.seed(42)

traffic_config = [
    {"version": "v1 (current)", "weight": 90},
    {"version": "v2 (candidate)", "weight": 10},
]

# Simulate 100 requests
v1_count = 0
v2_count = 0
for _ in range(100):
    if random.randint(1, 100) <= 90:
        v1_count += 1
    else:
        v2_count += 1

print("=== A/B Test Traffic Simulation (100 requests) ===")
print()
print(f"  Version 1 (current):   {v1_count} requests ({v1_count}%)")
print(f"  Version 2 (candidate): {v2_count} requests ({v2_count}%)")
print()
print("Traffic config sent to serving endpoint:")
for route in traffic_config:
    print(f"  {route['version']}: {route['weight']}%")
print()
print("KEY EXAM POINT: Traffic splitting lets you test a new model")
print("version in production without risking all traffic.")

### Expected Output

```
=== A/B Test Traffic Simulation (100 requests) ===

  Version 1 (current):   91 requests (91%)
  Version 2 (candidate): 9 requests (9%)

Traffic config sent to serving endpoint:
  v1 (current): 90%
  v2 (candidate): 10%

KEY EXAM POINT: Traffic splitting lets you test a new model
version in production without risking all traffic.
```

### What Just Happened

We demonstrated **A/B testing with traffic splitting**. In production, Databricks serving endpoints handle the routing automatically — you just configure the percentage. This is a safe way to validate a new model version before committing 100% of traffic.

---

## Step 6: Rollback Strategy

Rollback is the safety net of CI/CD. If a new model version performs worse than the current one, you need to quickly revert to the previous version.

In [ ]:
# ---- Step 6: Rollback strategy ----
# Rollback means shifting 100% of traffic back to the previous version.

# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
#
# # ROLLBACK: Route all traffic back to version 1
# w.serving_endpoints.update_config(
#     name="my-agent-endpoint",
#     traffic_config={
#         "routes": [
#             {"served_model_name": "my_agent-1", "traffic_percentage": 100},
#             {"served_model_name": "my_agent-2", "traffic_percentage": 0},
#         ]
#     }
# )

rollback_scenarios = [
    {
        "trigger": "Eval scores drop below threshold",
        "action": "Route 100% traffic to previous version",
        "how": "Update traffic_config to 100/0 split",
        "timeline": "Immediate (< 1 minute)"
    },
    {
        "trigger": "Error rate exceeds 5% on new version",
        "action": "Automatic rollback via monitoring alert",
        "how": "Databricks alert triggers a Workflow that updates traffic",
        "timeline": "Automated (< 5 minutes)"
    },
    {
        "trigger": "User complaints about quality regression",
        "action": "Manual rollback via UI or API",
        "how": "UI --> Serving --> Endpoint --> Edit traffic config",
        "timeline": "Manual (depends on detection time)"
    },
]

print("=== Rollback Strategies ===")
print()
for i, s in enumerate(rollback_scenarios, 1):
    print(f"  Scenario {i}: {s['trigger']}")
    print(f"    Action:   {s['action']}")
    print(f"    How:      {s['how']}")
    print(f"    Timeline: {s['timeline']}")
    print()

print("KEY EXAM POINT: Rollback is about traffic routing, not model deletion.")
print("The old version remains registered in UC -- you just redirect traffic.")

### Expected Output

```
=== Rollback Strategies ===

  Scenario 1: Eval scores drop below threshold
    Action:   Route 100% traffic to previous version
    How:      Update traffic_config to 100/0 split
    Timeline: Immediate (< 1 minute)

  Scenario 2: Error rate exceeds 5% on new version
    Action:   Automatic rollback via monitoring alert
    How:      Databricks alert triggers a Workflow that updates traffic
    Timeline: Automated (< 5 minutes)

  Scenario 3: User complaints about quality regression
    Action:   Manual rollback via UI or API
    How:      UI --> Serving --> Endpoint --> Edit traffic config
    Timeline: Manual (depends on detection time)

KEY EXAM POINT: Rollback is about traffic routing, not model deletion.
The old version remains registered in UC -- you just redirect traffic.
```

### What Just Happened

We covered three **rollback strategies** with different triggers and timelines. The key insight for the exam: rollback on Databricks is a **traffic routing change**, not a code revert. The old model version is still in Unity Catalog — you just point traffic back to it.

---

## Environment Promotion: Dev, Staging, Production

Databricks uses **Unity Catalog schemas** to separate environments. This is a critical exam concept.

```
main.genai_dev        -->  main.genai_staging     -->  main.genai_prod
(experiments,              (validation,                 (live traffic,
 prototyping)               eval tests)                  SLA-bound)
```

| Environment | UC Schema | Purpose | Who Has Access |
|-------------|-----------|---------|----------------|
| **Dev** | `main.genai_dev` | Experimentation, rapid iteration | Data scientists |
| **Staging** | `main.genai_staging` | Validation, evaluation tests | CI/CD pipeline |
| **Prod** | `main.genai_prod` | Live traffic, monitored | Serving endpoints only |

In [ ]:
# ---- Environment promotion: create schemas for dev/staging/prod ----
# In a real setup, these would be separate schemas with different permissions.

environments = [
    {"name": "dev",     "schema": "main.genai_dev",     "purpose": "Experimentation and prototyping"},
    {"name": "staging", "schema": "main.genai_staging", "purpose": "Validation and evaluation tests"},
    {"name": "prod",    "schema": "main.genai_prod",    "purpose": "Live traffic, SLA-bound"},
]

print("=== Environment Promotion Schemas ===")
print()
for env in environments:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env['schema']}")
    print(f"  [{env['name'].upper():<8}] {env['schema']:<25} -- {env['purpose']}")

print()
print("Promotion flow:")
print("  1. Data scientist logs model to main.genai_dev")
print("  2. CI/CD pipeline copies model to main.genai_staging")
print("  3. Eval tests pass --> model promoted to main.genai_prod")
print("  4. Serving endpoint updated to point at prod model version")

### Expected Output

```
=== Environment Promotion Schemas ===

  [DEV     ] main.genai_dev              -- Experimentation and prototyping
  [STAGING ] main.genai_staging          -- Validation and evaluation tests
  [PROD    ] main.genai_prod             -- Live traffic, SLA-bound

Promotion flow:
  1. Data scientist logs model to main.genai_dev
  2. CI/CD pipeline copies model to main.genai_staging
  3. Eval tests pass --> model promoted to main.genai_prod
  4. Serving endpoint updated to point at prod model version
```

### What Just Happened

We created the **three-environment promotion pattern** using Unity Catalog schemas. This is a best practice for GenAI CI/CD — each environment has its own schema, permissions, and purpose. Models flow from dev to staging to prod through automated gates.

---

## Exam Tips and Traps

| # | Tip or Trap | Explanation |
|---|-------------|-------------|
| 1 | **Trap:** "Deploy directly from notebooks" | Use MLflow Model Registry in Unity Catalog. Notebooks are for development. |
| 2 | **Tip:** CI/CD for GenAI has THREE test levels | Unit (code), integration (end-to-end), evaluation (output quality). |
| 3 | **Trap:** "One schema is enough for all environments" | Use separate schemas: `dev`, `staging`, `prod` with different permissions. |
| 4 | **Tip:** A/B testing = traffic splitting on serving endpoints | Configure percentage-based routing between model versions. |
| 5 | **Trap:** "Rollback means deleting the new model" | Rollback is a traffic routing change. The old version stays in UC. |
| 6 | **Tip:** Databricks Workflows orchestrate the pipeline | Jobs trigger on Git push, run tests, deploy models in sequence. |

---

## Key Takeaways

### Core Concepts

- **GenAI CI/CD** adds evaluation tests (LLM judges) on top of traditional unit and integration tests
- **Databricks Repos** provides Git integration for version-controlling agents, chains, and tools
- **Databricks Workflows** orchestrate the pipeline: test, log, register, deploy, evaluate
- **MLflow Model Registry (UC)** is the artifact store — all model versions, lineage, and metadata
- **Traffic splitting** enables safe A/B testing between model versions
- **Rollback** is a traffic routing change, not a code revert

### Pipeline Summary

```
Git Push --> Unit Tests --> Integration Tests --> MLflow Log
    --> UC Register --> Staging Deploy --> Eval Tests --> Production Deploy
```

### Environment Promotion

```
main.genai_dev  -->  main.genai_staging  -->  main.genai_prod
  (experiment)         (validate)               (serve)
```

---

## Cleanup (Optional)

Uncomment and run the following cell to remove the schemas created in this lab.

In [ ]:
# ---- Cleanup: Remove lab schemas (uncomment to run) ----
# spark.sql("DROP SCHEMA IF EXISTS main.genai_dev CASCADE")
# spark.sql("DROP SCHEMA IF EXISTS main.genai_staging CASCADE")
# spark.sql("DROP SCHEMA IF EXISTS main.genai_prod CASCADE")
# print("Cleanup complete.")

---

## Next Lab

Continue to **Lab 04-07: MCP Servers** (`07_mcp_servers.ipynb`), where you will learn about Model Context Protocol (MCP) servers — managed, external, and custom — that give agents access to external tools and data sources.